# Retail Demand Forecasting, Stock-out Risk and Inventory Strategy
**Tredence brief, Task 1: Retail Demand Forecasting**

**Business problem:** A retailer needs to know how many units of each item each store will sell over the next quarter, how likely it is to run out of stock while waiting for a delivery, and how much safety stock to hold.

**Type:** time-series forecasting (regression) + risk scoring  |  **Audience:** supply-chain and category managers  |  **Outcome:** 90-day forecast per store-item, stock-out probability, reorder points, ABC inventory policy

**Data:** Kaggle *Store Item Demand Forecasting*: daily sales, 10 stores x 50 items, 2013-01-01 to 2017-12-31 (913,000 rows). Task: forecast 2018-01-01 to 2018-03-31.

**Metrics:** SMAPE (the competition metric), MAE, RMSE, WAPE. **Validation:** two rolling back-tests on the last two quarters of known data (time-ordered, no shuffling).

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from forecasting import *

sns.set_theme(style="whitegrid")

## 2. Load data and quality checks

In [ ]:
df = load_sales()
print(df.shape, df["date"].min().date(), "to", df["date"].max().date())
print("stores:", df["store"].nunique(), "| items:", df["item"].nunique())
print("missing values:", int(df.isna().sum().sum()), "| duplicate rows:", int(df.duplicated().sum()))
print("negative sales:", int((df["sales"] < 0).sum()), "| zero sales:", int((df["sales"] == 0).sum()))
expected = df["store"].nunique() * df["item"].nunique() * df["date"].nunique()
print("complete grid (every series has every day):", len(df) == expected)
df.dtypes

**Cleaning decisions:** the data is already clean: no missing values, duplicates or negatives, and every store-item has a full daily history, so no imputation or outlier removal is needed. Dates are parsed to `datetime`. Zero-sales days are real (not errors) and are kept; SMAPE treats a 0/0 day as zero error.

## 3. Exploratory data analysis

In [ ]:
# Univariate: total demand over time and distribution of daily sales
total = df.groupby("date")["sales"].sum()
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
total.plot(ax=ax[0], lw=0.6); total.rolling(30).mean().plot(ax=ax[0], lw=2, color="red")
ax[0].set_title("Total daily sales (red = 30-day average)")
sns.histplot(df["sales"], bins=60, ax=ax[1]); ax[1].set_title("Distribution of daily sales per store-item")
plt.tight_layout(); plt.show()
print(df["sales"].describe().round(1))

In [ ]:
# Seasonality: day of week, month, year trend
d = df.assign(dow=df["date"].dt.day_name(), month=df["date"].dt.month, year=df["date"].dt.year)
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
d.groupby("dow")["sales"].mean().reindex(dow_order).plot.bar(ax=ax[0], title="Average sales by weekday")
d.groupby("month")["sales"].mean().plot.bar(ax=ax[1], title="Average sales by month")
d.groupby("year")["sales"].mean().plot.bar(ax=ax[2], title="Average sales by year")
plt.tight_layout(); plt.show()
yearly = d.groupby("year")["sales"].mean(); print((yearly.pct_change() * 100).round(1).dropna().rename("YoY growth %"))

In [ ]:
# Bivariate / multivariate: stores x items
fig, ax = plt.subplots(1, 2, figsize=(17, 5))
df.groupby("store")["sales"].mean().sort_values().plot.barh(ax=ax[0], title="Average daily sales by store")
pivot = df.pivot_table(index="store", columns="item", values="sales", aggfunc="mean")
sns.heatmap(pivot, cmap="viridis", ax=ax[1], cbar_kws={"label": "avg daily sales"}); ax[1].set_title("Store x item average sales")
plt.tight_layout(); plt.show()
print("Item range (avg daily sales):", df.groupby("item")["sales"].mean().agg(["min", "max"]).round(1).to_dict())

In [ ]:
# Do all series share the same seasonal shape? (month index per item, scaled to its own mean)
idx = df.assign(month=df["date"].dt.month).groupby(["item", "month"])["sales"].mean().unstack()
idx = idx.div(idx.mean(axis=1), axis=0)
plt.figure(figsize=(11, 4)); plt.plot(idx.T, color="grey", alpha=0.3); plt.plot(idx.mean(), color="red", lw=3)
plt.title("Monthly seasonal index per item (red = average): shapes are very similar"); plt.xlabel("month"); plt.show()

**EDA takeaways**
- Demand grows year on year and has a strong **annual seasonality** (peak in summer, trough in winter) and a strong **weekly pattern** (weekend high).
- Stores differ mainly in **level**; items differ in level too (roughly a 5x range), but they share the **same seasonal shape**. That justifies one *global* model for all 500 series instead of 500 separate models.
- Because last year's same-weekday sales and the current level carry most of the signal, the features below are built from those.

## 4. Feature engineering
Everything is built from the (series x day) sales matrix. **Each feature for day t uses only sales from day t-90 or earlier**, where 90 is the forecast horizon. This lets one model forecast the whole quarter at once (no recursive forecasting, no error build-up) and rules out look-ahead leakage by construction.

| Feature | Meaning |
|---|---|
| store, item, dow, month, dom, woy, doy | identity and calendar |
| lag_h, dow_mean_4w | sales 90 days ago; average of the same weekday over the last 4 weeks of known data |
| ly_same_day, ly_smooth, lag_728 | same weekday last year (and a 3-week smoothed version), two years ago |
| roll_7/28/91/365 | rolling mean sales ending 90 days before the target day (current level) |
| yoy_growth | roll_365 now vs a year earlier (growth trend) |
| ly_ratio | last year's same-period level vs the recent level (seasonal index) |

In [ ]:
Y, series, dates = to_matrix(df)
feat = build_features(Y, series, dates)
print(feat.shape)
feat[FEATURES].describe().T[["count", "mean", "min", "max"]].round(2)

## 5. Models and validation
- **Weekday average:** mean of the same weekday over the last 4 known weeks (simple baseline).
- **Seasonal naive x growth:** same weekday last year, smoothed, scaled by year-on-year growth (strong business baseline).
- **Gradient boosting (HistGradientBoostingRegressor):** one global model on the features above, trained on log(1+sales).

Back-test fold B below (train up to 2017-09-30, forecast the next 90 days). The saved two-fold averages come from `python src/train.py`.

In [ ]:
cut = pd.Timestamp("2017-09-30")
model = fit_model(feat, cut)
v = fold_rows(feat, cut)
v["hgb"] = predict(model, v)
v["snaive"] = seasonal_naive(v)
v["dow4w"] = v["dow_mean_4w"].fillna(v["roll_91"]).values
print("Validation window:", v["date"].min().date(), "to", v["date"].max().date())
fold_b = pd.DataFrame({n: metrics(v["sales"], v[c]) for n, c in
                       [("Weekday average (4 wk)", "dow4w"), ("Seasonal naive x growth", "snaive"), ("Gradient boosting", "hgb")]}).T
fold_b

In [ ]:
# Two-fold average (from the full pipeline run)
pd.read_csv(ROOT / "reports" / "model_comparison.csv", index_col=0)

**Model selection:** gradient boosting beats both baselines on every metric, in both folds. It reduces error by roughly 15% in SMAPE vs the strong seasonal-naive baseline, and about 47% vs the weekday average. It is also fast (about 30 seconds to train) and handles 500 series with one model. The ARIMA and LSTM models requested in the brief are in `02_arima_lstm_optional.ipynb`.

## 6. Evaluation

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7))
daily = v.groupby("date")[["sales", "hgb", "snaive"]].sum()
daily.plot(ax=ax[0], title="Total demand: actual vs forecast (unseen quarter)")
g = v[(v["store"] == 1) & (v["item"] == 1)]
ax[1].plot(g["date"], g["sales"], label="actual"); ax[1].plot(g["date"], g["hgb"], label="forecast")
ax[1].set_title("Store 1, item 1"); ax[1].legend(); plt.tight_layout(); plt.show()
print("SMAPE of total daily demand (all series summed): HGB %.2f%%, seasonal naive %.2f%%" %
      (smape(daily["sales"], daily["hgb"]), smape(daily["sales"], daily["snaive"])))

In [ ]:
# Where is the error? SMAPE by store, by item, by weekday
v["ape"] = 200 * (v["sales"] - v["hgb"]).abs() / (v["sales"] + v["hgb"]).replace(0, np.nan)
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
v.groupby("store")["ape"].mean().plot.bar(ax=ax[0], title="SMAPE by store")
v.groupby("item")["ape"].mean().plot.bar(ax=ax[1], title="SMAPE by item")
v.groupby("dow")["ape"].mean().plot.bar(ax=ax[2], title="SMAPE by weekday (0 = Monday)")
plt.tight_layout(); plt.show()

In [ ]:
# Residuals and feature importance
resid = v["sales"] - v["hgb"]
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
sns.histplot(resid, bins=60, ax=ax[0]); ax[0].set_title("Forecast error (actual - forecast), units")
imp = pd.read_csv(ROOT / "reports" / "feature_importance.csv", index_col=0)["importance"]
imp.head(10).iloc[::-1].plot.barh(ax=ax[1], title="Permutation importance (top 10)")
plt.tight_layout(); plt.show()
print("mean error (bias): %.2f units | std: %.2f" % (resid.mean(), resid.std()))

**Evaluation takeaways:** the aggregate forecast is very accurate (total demand within about 1% daily), while per-series error is higher because individual store-item sales are noisy (roughly Poisson). Last year's same-weekday level (`ly_smooth`) is by far the most important feature, followed by month and weekday: seasonality drives this data.

## 7. Stock-out risk
Forecast error tells us how uncertain demand is. For a supplier lead time of L days:

- expected lead-time demand  mu_L = sum of the L daily forecasts
- uncertainty  sigma_L = c_L x sigma_d x sqrt(L), where sigma_d is the per-series daily forecast error and c_L a calibration factor measured on back-tests (it is close to 1)
- **safety stock** = z x sigma_L, with z = the normal quantile of the target service level
- **reorder point** = mu_L + safety stock
- **stock-out probability** for stock S = P(demand > S) = 1 - Phi((S - mu_L) / sigma_L)

**Back-test:** sigma_d and c_L were calibrated on fold A and the policy was tested on unseen fold B: did the share of lead-time windows without a stock-out match the target?

In [ ]:
bt = pd.read_csv(ROOT / "reports" / "stockout_backtest.csv"); bt

The policy **meets or slightly exceeds each target** on unseen data (for example a 95% target gave about 96.5% of windows without a stock-out), so it is mildly conservative rather than risky. The normal approximation is therefore reasonable for this data.

## 8. Inventory strategy (ABC policy)

In [ ]:
stats = pd.read_csv(ROOT / "reports" / "series_stats.csv")
fc = pd.read_csv(ROOT / "data" / "forecasts_2018Q1.csv", parse_dates=["date"])
summary = stats.groupby("abc").agg(series=("item", "size"), avg_daily_units=("avg_daily_2017", "mean"))
summary["share_of_volume_%"] = (stats.groupby("abc")["avg_daily_2017"].sum() / stats["avg_daily_2017"].sum() * 100).round(1)
summary.round(1)

In [ ]:
import json
print(json.dumps(json.load(open(ROOT / "reports" / "metrics.json"))["inventory_policy"], indent=2))

**Recommended policy**
- **Class A** (311 store-items, about 80% of volume): 98% service level, review weekly, reorder at the computed reorder point.
- **Class B** (123, about 15%): 95% service level.
- **Class C** (66, about 5%): 90% service level; accept occasional stock-outs rather than tie up cash.
- This policy holds about **16% more safety stock units** than a flat 95% rule (20,367 vs 17,537 units for 7-day lead times), but concentrates protection on the items that make up most of sales. Holding costs and lost-sales costs are not in the dataset, so the right trade-off has to be set with the client.
- Re-forecast monthly and re-check back-test error; raise safety stock before known promotions (use the demand what-if in the app).

## 9. Forecast for 2018 Q1

In [ ]:
tot = fc.groupby("date")["forecast"].sum()
hist_tot = df[df["date"] >= "2017-01-01"].groupby("date")["sales"].sum()
plt.figure(figsize=(12, 4)); plt.plot(hist_tot, label="2017 actual"); plt.plot(tot, label="2018 Q1 forecast", color="red")
plt.title("Total daily demand: history and forecast"); plt.legend(); plt.show()
print("Forecast rows:", len(fc), "| Q1 2018 total units: %.0f" % fc["forecast"].sum(),
      "| Q1 2017 actual: %.0f" % df[(df["date"] >= "2017-01-01") & (df["date"] <= "2017-03-31")]["sales"].sum())

## 10. Conclusions and next steps
1. A single global gradient-boosting model forecasts 500 store-item series with about **11.7% SMAPE** (two back-tests), versus 13.8% for a seasonal-naive baseline and 22.1% for a weekday average.
2. Demand is dominated by annual and weekly seasonality plus steady growth; store and item differ mainly in level.
3. Stock-out risk and safety stock follow directly from forecast error and were validated on unseen data: targets were met or slightly exceeded.
4. **Limitations:** no price, promotion, holiday or weather data; fixed lead time; normal approximation; no cost data.
5. **Next:** add holiday/promotion features, quantile models for asymmetric costs, ARIMA/LSTM comparison (see optional notebook), and a monthly retraining job.